In [1]:

!CONDA_BASE="$HOME/miniconda3"
!source "$CONDA_BASE/etc/profile.d/conda.sh"

!conda env list
!which conda
import sys
print("Python path:", sys.executable)
print("Python version:", sys.version)
# Check for GPU with PyTorch
import torch
print(torch.cuda.is_available())

# Check for GPU with TensorFlow
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

import tensorflow as tf
print(tf.__version__)  # Should show 2.15.x
print("GPU Available:", tf.config.list_physical_devices('GPU'))

/bin/bash: line 1: /etc/profile.d/conda.sh: No such file or directory

# conda environments:
#
nnunet_env             /home/rbielski/.conda/envs/nnunet_env
pycharm_env            /home/rbielski/.conda/envs/pycharm_env
stroke_sota            /home/rbielski/.conda/envs/stroke_sota
tf215_env              /home/rbielski/.conda/envs/tf215_env
tf_2_15                /home/rbielski/.conda/envs/tf_2_15
base                   /home/rbielski/miniconda3
geo_env                /home/rbielski/miniconda3/envs/geo_env
stroke_env           * /home/rbielski/miniconda3/envs/stroke_env
tf215_env_recreated    /home/rbielski/miniconda3/envs/tf215_env_recreated

/home/rbielski/miniconda3/condabin/conda
Python path: /home/rbielski/miniconda3/envs/stroke_env/bin/python
Python version: 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]
True


2025-09-09 15:56:16.566839: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-09 15:56:16.566872: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-09 15:56:16.567916: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-09 15:56:16.572842: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-09 15:56:17.187498: W tensorflow/compiler/tf2

[]
2.15.0
GPU Available: []


2025-09-09 15:56:17.758699: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-09-09 15:56:17.760348: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-09-09 15:56:17.762357: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required l

In [9]:
# 📊 ROBUST DATASET PREPARATION with CROPPED_COMBINED Atlas Data (self-contained)
import numpy as np
from pathlib import Path

print("🔍 Setting up dataset loading...")

# Minimal local config (no external training module needed)
class TrainingConfig:
    def __init__(self):
        self.DATA_DIR = Path("/home/rbielski/Atlas_2/Training/Cropped_128_Combined")
        self.INPUT_SHAPE = (128, 128, 128, 1)

# Create or ensure config exists
if 'config' not in globals() or config is None:
    config = TrainingConfig()
    print("📝 Created new config object")

# Ensure DATA_DIR is a Path object
if not hasattr(config, 'DATA_DIR') or config.DATA_DIR is None:
    config.DATA_DIR = Path("/home/rbielski/Atlas_2/Training/Cropped_128_Combined")
else:
    config.DATA_DIR = Path(config.DATA_DIR)

# Helper: detect flat combined dataset (no Images/Masks subfolders)
def has_flat_combined_dir(p: Path) -> bool:
    if not p.exists():
        return False
    imgs = list(p.glob("*nii_T1w_cropped128.nii.gz"))
    msks = list(p.glob("*nii_mask_cropped128.nii.gz"))
    return len(imgs) > 0 and len(msks) > 0

CROPPED_DEFAULT = Path("/home/rbielski/Atlas_2/Training/Cropped_128_Combined")

# Find a valid data directory (PRIORITIZE flat Cropped_128_Combined)
fallback_paths = [
    CROPPED_DEFAULT,
    Path("../Atlas_2/Training/Cropped_128_Combined"),
    Path("../../Atlas_2/Training/Cropped_128_Combined"),
    Path("./Atlas_2/Training/Cropped_128_Combined"),
    config.DATA_DIR,  # last: respect prior config if above didn't match flat
]

data_dir_found = None
layout = None  # 'flat' or 'split'
for candidate_path in fallback_paths:
    if not candidate_path.exists():
        print(f"✗ Not found: {candidate_path}")
        continue
    if has_flat_combined_dir(candidate_path):
        data_dir_found = candidate_path
        layout = 'flat'
        print(f"✅ Using flat combined dataset at: {data_dir_found}")
        break
    if (candidate_path / "Images").exists() and (candidate_path / "Masks").exists():
        data_dir_found = candidate_path
        layout = 'split'
        print(f"✅ Using split dataset (Images/Masks) at: {data_dir_found}")
        break
    print(f"✗ Not matching known layouts: {candidate_path}")

if data_dir_found is None:
    print("❌ No valid Atlas data directory found!")
    print("Expected either flat files '*T1w_cropped128.nii.gz' + '*mask_cropped128.nii.gz' or split Images/ + Masks/ folders")
    pairs, lesion_presence = [], []
else:
    # Update config to the selected directory
    config.DATA_DIR = data_dir_found

    print(f"📚 Loading dataset manually (layout={layout})...")
    if layout == 'split':
        images_dir = data_dir_found / "Images"
        masks_dir = data_dir_found / "Masks"
        image_patterns = ['*nii_T1w_cropped128.nii.gz', '*_t1.nii.gz', '*T1w.nii.gz', '*t1w.nii.gz']
        mask_patterns = ['*nii_mask_cropped128.nii.gz', '*_lesion.nii.gz', '*_label-L*.nii.gz']
        images = []
        for pattern in image_patterns:
            found_images = list(images_dir.glob(pattern))
            if found_images:
                images.extend(found_images)
                break
        masks = []
        for pattern in mask_patterns:
            found_masks = list(masks_dir.glob(pattern))
            if found_masks:
                masks.extend(found_masks)
                break
    else:  # flat combined
        images = sorted(data_dir_found.glob("*nii_T1w_cropped128.nii.gz"))
        masks = sorted(data_dir_found.glob("*nii_mask_cropped128.nii.gz"))

    # Build key maps for robust pairing
    def basekey_from_name(name: str) -> str:
        if name.endswith("nii_T1w_cropped128.nii.gz"):
            return name[:-len("nii_T1w_cropped128.nii.gz")]
        if name.endswith("nii_mask_cropped128.nii.gz"):
            return name[:-len("nii_mask_cropped128.nii.gz")]
        # Generic fallback: before first .nii
        return name.split(".nii")[0]

    image_map = {basekey_from_name(p.name): p for p in images}
    mask_map = {basekey_from_name(p.name): p for p in masks}

    # Intersect keys to form pairs
    common_keys = sorted(set(image_map.keys()) & set(mask_map.keys()))
    pairs = [(image_map[k], mask_map[k]) for k in common_keys]
    lesion_presence = [1] * len(pairs)

    # Report unmatched files (useful for debugging naming mismatches)
    missing_masks = sorted(k for k in image_map.keys() if k not in mask_map)
    missing_images = sorted(k for k in mask_map.keys() if k not in image_map)
    if missing_masks:
        print(f"ℹ️ Images without masks: {len(missing_masks)} (e.g., {missing_masks[:3]})")
    if missing_images:
        print(f"ℹ️ Masks without images: {len(missing_images)} (e.g., {missing_images[:3]})")

    print(f"✅ Manually paired {len(pairs)} image-mask pairs")
    if len(pairs) > 0:
        from pathlib import Path as _P
        print("   Examples:")
        for i, (img, msk) in enumerate(pairs[:3]):
            print(f"   {i+1:>2}. {_P(img).name}  <->  {_P(msk).name}")

# Summary
if len(pairs) > 0:
    print(f"\n📊 DATASET SUMMARY:")
    print(f"   Total pairs: {len(pairs)}")
    if len(lesion_presence) > 0:
        print(f"   Lesion presence: {np.mean(lesion_presence)*100:.1f}% of samples")
    print(f"   Data directory: {config.DATA_DIR}")
    try:
        from pathlib import Path as _P
        print(f"   Example pair: {_P(pairs[0][0]).name} <-> {_P(pairs[0][1]).name}")
    except Exception:
        pass
else:
    print(f"\n❌ No dataset pairs loaded!")

print(f"\nDataset ready: {len(pairs)} pairs available for testing")

🔍 Setting up dataset loading...
✅ Using flat combined dataset at: /home/rbielski/Atlas_2/Training/Cropped_128_Combined
📚 Loading dataset manually (layout=flat)...
✅ Manually paired 655 image-mask pairs
   Examples:
    1. sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz  <->  sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_mask_cropped128.nii.gz
    2. sub-r001s002_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz  <->  sub-r001s002_ses-1_space-MNI152NLin2009aSym.nii_mask_cropped128.nii.gz
    3. sub-r001s003_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz  <->  sub-r001s003_ses-1_space-MNI152NLin2009aSym.nii_mask_cropped128.nii.gz

📊 DATASET SUMMARY:
   Total pairs: 655
   Lesion presence: 100.0% of samples
   Data directory: /home/rbielski/Atlas_2/Training/Cropped_128_Combined
   Example pair: sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz <-> sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_mask_cropped128.nii.gz

Dataset re

In [2]:
# Robust Model Loader with Custom Object Registration (CROPPED 128x128x128)
import sys
from pathlib import Path
import tensorflow as tf

# Ensure we can import the cropped training module with custom layers
BASE_DIR_CANDIDATES = [
    Path.cwd(),
    Path('/home/rbielski/stroke_cleaned/stroke_segmentation_v1.1_success')
]
for _b in BASE_DIR_CANDIDATES:
    if (_b / 'smart_sota_2025_claude_cropped.py').exists():
        if str(_b) not in sys.path:
            sys.path.insert(0, str(_b))
        break

try:
    from smart_sota_2025_claude_cropped import (
        ResidualConvBlock, VisionMambaBlock, SAM2Attention,
        dice_coefficient as dice_coefficient_tf,
        boundary_weighted_loss as boundary_weighted_loss_tf
    )
    print('✅ Imported custom layers from smart_sota_2025_claude_cropped.py')
except Exception as e:
    print(f'⚠️ Could not import cropped custom layers: {e}')
    ResidualConvBlock = None
    VisionMambaBlock = None
    SAM2Attention = None
    dice_coefficient_tf = None
    boundary_weighted_loss_tf = None

def create_model_shims():
    def dice_coeff_shim(y_true, y_pred):
        y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
        intersection = tf.reduce_sum(y_true_f * y_pred_f)
        return (2. * intersection + 1e-7) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + 1e-7)
    def boundary_loss_shim(y_true, y_pred):
        return 1.0 - dice_coeff_shim(y_true, y_pred)
    return dice_coeff_shim, boundary_loss_shim

dice_shim, boundary_shim = create_model_shims()

custom_objects = {
    'ResidualConvBlock': ResidualConvBlock,
    'VisionMambaBlock': VisionMambaBlock,
    'SAM2Attention': SAM2Attention,
    'dice_coefficient': dice_coefficient_tf or dice_shim,
    'boundary_weighted_loss': boundary_weighted_loss_tf or boundary_shim,
    'compiled_loss': boundary_weighted_loss_tf or boundary_shim,
    'loss': boundary_weighted_loss_tf or boundary_shim
}
tf.keras.utils.get_custom_objects().update(custom_objects)

# Discover cropped model files (add explicit known path first)
search_roots = [p for p in BASE_DIR_CANDIDATES if p.exists()]
model_candidates = []

# 1) Explicit absolute path provided by README/user
explicit_path = Path('/home/rbielski/stroke_cleaned/stroke_segmentation_v1.1_success/models/emergency_cropped_save_20250721_193525.keras')
model_candidates.append(explicit_path)

# 2) Common checkpoint locations under repo roots
for root in search_roots:
    model_candidates.append(root / 'callbacks/cropped128_production/best_model_cropped128.keras')
    # Any .keras under cropped models dir
    models_dir = root / 'models/cropped128_production'
    if models_dir.exists():
        try:
            found = sorted(models_dir.glob('*.keras'), key=lambda x: x.stat().st_mtime, reverse=True)
            model_candidates.extend(found)
        except Exception:
            pass
    # Also scan generic models/ for .keras files
    generic_models = root / 'models'
    if generic_models.exists():
        try:
            found_generic = sorted(generic_models.glob('*.keras'), key=lambda x: x.stat().st_mtime, reverse=True)
            model_candidates.extend(found_generic)
        except Exception:
            pass

m = None
for model_path in model_candidates:
    if model_path.exists():
        print(f'Found model candidate: {model_path}')
        try:
            print('Attempting model loading with custom objects...')
            m = tf.keras.models.load_model(str(model_path), custom_objects=custom_objects, compile=False)
            if m is not None:
                print('✅ Model loaded successfully')
                print(f'Model input shape: {m.input_shape}')
                print(f'Model output shape: {m.output_shape}')
                break
        except Exception as e:
            print(f'Failed to load model from {model_path}: {e}')
            import traceback; traceback.print_exc()
    else:
        print(f'Model file not found: {model_path}')

if m is None:
    raise FileNotFoundError('No valid cropped model file found. Please verify the explicit_path above exists.')

print("✅ Model 'm' is ready for inference (cropped 128^3)")

✅ Imported custom layers from smart_sota_2025_claude_cropped.py
Found model candidate: /home/rbielski/stroke_cleaned/stroke_segmentation_v1.1_success/models/emergency_cropped_save_20250721_193525.keras
Attempting model loading with custom objects...
✅ Model loaded successfully
Model input shape: (None, 128, 128, 128, 1)
Model output shape: (None, 128, 128, 128, 1)
✅ Model 'm' is ready for inference (cropped 128^3)
✅ Model loaded successfully
Model input shape: (None, 128, 128, 128, 1)
Model output shape: (None, 128, 128, 128, 1)
✅ Model 'm' is ready for inference (cropped 128^3)


In [4]:
# ===================================================================
# 🧪 COMPLETE TESTING & VISUALIZATION PIPELINE (auto-shape from model)
# ===================================================================

print('🧪 Setting up comprehensive testing pipeline (cropped-aware)...')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets
from IPython.display import display
import nibabel as nib
import logging

# Suppress nibabel logging
logging.getLogger('nibabel').setLevel(logging.WARNING)

# Determine target input shape from loaded model (fallback to 128^3)
MODEL_INPUT_SHAPE = None
try:
    if 'm' in globals() and hasattr(m, 'input_shape') and m.input_shape and len(m.input_shape) == 5:
        MODEL_INPUT_SHAPE = tuple(int(x) for x in m.input_shape[1:4])
except Exception:
    pass
if MODEL_INPUT_SHAPE is None:
    MODEL_INPUT_SHAPE = (128, 128, 128)
print(f'🎯 Using target input shape: {MODEL_INPUT_SHAPE}')

# Create test results directory
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
test_results_dir = Path('test_results')
test_results_dir.mkdir(exist_ok=True)

def dice_coefficient_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def iou_score_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def robust_resize_with_context(volume, target_shape=None, preserve_dtype=True):
    target_shape = tuple(target_shape or MODEL_INPUT_SHAPE)
    try:
        from scipy.ndimage import zoom
        if volume is None:
            raise ValueError('Volume is None')
        original_shape = volume.shape
        original_dtype = volume.dtype if preserve_dtype else None
        zoom_factors = [t/o for t, o in zip(target_shape, original_shape)]
        resized = zoom(volume, zoom_factors, order=1, prefilter=True)
        if preserve_dtype and original_dtype is not None:
            if original_dtype == np.bool_ or 'bool' in str(original_dtype):
                resized = (resized > 0.5).astype(np.bool_)
            else:
                resized = resized.astype(original_dtype)
        return resized
    except Exception as e:
        print(f'❌ Resize error: {e}')
        fallback = np.zeros(target_shape, dtype=volume.dtype if volume is not None else np.float32)
        return fallback

def preprocess_volume(volume_path, target_shape=None):
    target_shape = tuple(target_shape or MODEL_INPUT_SHAPE)
    try:
        nii = nib.load(str(volume_path))
        volume = nii.get_fdata()
        if volume.ndim == 4:
            volume = volume[:, :, :, 0]
        elif volume.ndim != 3:
            raise ValueError(f'Unexpected volume dimensions: {volume.shape}')
        volume_resized = robust_resize_with_context(volume, target_shape)
        if volume_resized.max() > volume_resized.min():
            volume_resized = (volume_resized - volume_resized.min()) / (volume_resized.max() - volume_resized.min())
        volume_final = volume_resized[np.newaxis, ..., np.newaxis]
        return volume_final
    except Exception as e:
        print(f'❌ Error preprocessing {volume_path}: {e}')
        fallback = np.zeros((1,) + tuple(target_shape) + (1,), dtype=np.float32)
        return fallback

def run_comprehensive_test(model, pairs, max_samples=None):
    if max_samples is None:
        max_samples = len(pairs)
    print(f'🚀 Starting comprehensive test on {min(len(pairs), max_samples)} samples...')
    results = {
        'timestamp': timestamp,
        'total_samples': min(len(pairs), max_samples),
        'model_info': {
            'input_shape': str(model.input_shape),
            'output_shape': str(model.output_shape),
            'parameters': model.count_params()
        },
        'sample_results': []
    }
    test_pairs = pairs[:max_samples]
    for i, (img_path, mask_path) in enumerate(test_pairs):
        print(f'📊 Processing sample {i+1}/{len(test_pairs)}: {Path(img_path).name}')
        try:
            img_volume = preprocess_volume(img_path, MODEL_INPUT_SHAPE)
            mask_nii = nib.load(str(mask_path))
            mask_volume = mask_nii.get_fdata()
            if mask_volume.ndim == 4:
                mask_volume = mask_volume[:, :, :, 0]
            mask_resized = robust_resize_with_context(mask_volume, MODEL_INPUT_SHAPE)
            mask_binary = (mask_resized > 0.5).astype(np.float32)
            prediction = model.predict(img_volume, verbose=0)
            pred_binary = (prediction[0, ..., 0] > 0.5).astype(np.float32)
            dice = dice_coefficient_np(mask_binary, pred_binary)
            iou = iou_score_np(mask_binary, pred_binary)
            gt_volume = np.sum(mask_binary)
            pred_volume = np.sum(pred_binary)
            volume_error = abs(pred_volume - gt_volume) / (gt_volume + 1e-6)
            sample_result = {
                'sample_id': i,
                'image_file': Path(img_path).name,
                'mask_file': Path(mask_path).name,
                'dice_score': float(dice),
                'iou_score': float(iou),
                'gt_volume': float(gt_volume),
                'pred_volume': float(pred_volume),
                'volume_error': float(volume_error),
                'prediction_shape': str(prediction.shape),
                'input_shape': str(img_volume.shape)
            }
            results['sample_results'].append(sample_result)
            pred_filename = f'prediction_sample_{i:03d}_{timestamp}.npy'
            np.save(test_results_dir / pred_filename, prediction[0, ..., 0])
            viz_filename = f'visualization_sample_{i:03d}_{timestamp}.png'
            save_prediction_visualization(img_volume[0, ..., 0], mask_binary, pred_binary, test_results_dir / viz_filename, sample_result)
            print(f'  ✅ Dice: {dice:.4f}, IoU: {iou:.4f}, Vol Error: {volume_error:.4f}')
        except Exception as e:
            print(f'  ❌ Error processing sample {i}: {e}')
            sample_result = {
                'sample_id': i,
                'image_file': Path(img_path).name,
                'mask_file': Path(mask_path).name,
                'error': str(e),
                'dice_score': 0.0,
                'iou_score': 0.0
            }
            results['sample_results'].append(sample_result)
    valid_results = [r for r in results['sample_results'] if 'error' not in r]
    if valid_results:
        dice_scores = [r['dice_score'] for r in valid_results]
        iou_scores = [r['iou_score'] for r in valid_results]
        results['summary'] = {
            'valid_samples': len(valid_results),
            'failed_samples': len(results['sample_results']) - len(valid_results),
            'mean_dice': float(np.mean(dice_scores)),
            'std_dice': float(np.std(dice_scores)),
            'mean_iou': float(np.mean(iou_scores)),
            'std_iou': float(np.std(iou_scores)),
            'min_dice': float(np.min(dice_scores)),
            'max_dice': float(np.max(dice_scores))
        }
    results_filename = f'test_results_{timestamp}.json'
    with open(test_results_dir / results_filename, 'w') as f:
        json.dump(results, f, indent=2)
    print('\n📊 TESTING COMPLETE!')
    print(f'📁 Results saved to: {test_results_dir}')
    print(f'📄 JSON report: {results_filename}')
    if 'summary' in results:
        summary = results['summary']
        print('\n📈 SUMMARY STATISTICS:')
        print(f"   Valid samples: {summary['valid_samples']}/{results['total_samples']}")
        print(f"   Mean Dice: {summary['mean_dice']:.4f} ± {summary['std_dice']:.4f}")
        print(f"   Mean IoU: {summary['mean_iou']:.4f} ± {summary['std_iou']:.4f}")
        print(f"   Dice range: [{summary['min_dice']:.4f}, {summary['max_dice']:.4f}]")
    return results

def save_prediction_visualization(image, ground_truth, prediction, filename, metrics):
    gt_slices = np.sum(ground_truth, axis=(0, 1))
    middle_slice = np.argmax(gt_slices) if np.max(gt_slices) > 0 else ground_truth.shape[2] // 2
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image[:, :, middle_slice], cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    axes[1].imshow(image[:, :, middle_slice], cmap='gray', alpha=0.7)
    axes[1].imshow(ground_truth[:, :, middle_slice], cmap='Reds', alpha=0.5)
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    axes[2].imshow(image[:, :, middle_slice], cmap='gray', alpha=0.7)
    axes[2].imshow(prediction[:, :, middle_slice], cmap='Blues', alpha=0.5)
    axes[2].set_title('Prediction')
    axes[2].axis('off')
    metrics_text = f"Dice: {metrics['dice_score']:.3f}\\nIoU: {metrics['iou_score']:.3f}"
    fig.suptitle(f"Sample {metrics['sample_id']} - {metrics_text}", fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()

print('✅ Testing pipeline ready for cropped input!')
print('📋 Available functions:')
print('   • run_comprehensive_test(model, pairs, max_samples=None)')
print('   • preprocess_volume(path, target_shape=None)')
print('   • save_prediction_visualization(...)')
print('   • dice_coefficient_np(y_true, y_pred)')
print('   • iou_score_np(y_true, y_pred)')

🧪 Setting up comprehensive testing pipeline (cropped-aware)...
🎯 Using target input shape: (128, 128, 128)
✅ Testing pipeline ready for cropped input!
📋 Available functions:
   • run_comprehensive_test(model, pairs, max_samples=None)
   • preprocess_volume(path, target_shape=None)
   • save_prediction_visualization(...)
   • dice_coefficient_np(y_true, y_pred)
   • iou_score_np(y_true, y_pred)


In [6]:
# ===================================================================
# 🚀 RUN COMPREHENSIVE TESTING ON REAL ATLAS DATA
# ===================================================================

# Ensure dataset 'pairs' exists; if not, do a quick discovery
from pathlib import Path
import numpy as np
if 'pairs' not in globals():
    print("⚠️ 'pairs' not found. Attempting quick dataset discovery...")
    def _has_flat_combined_dir(p: Path) -> bool:
        return p.exists() and any(p.glob("*nii_T1w_cropped128.nii.gz")) and any(p.glob("*nii_mask_cropped128.nii.gz"))
    CROPPED_DEFAULT = Path("/home/rbielski/Atlas_2/Training/Cropped_128_Combined")
    fallback_paths = [
        CROPPED_DEFAULT,
        Path("../Atlas_2/Training/Cropped_128_Combined"),
        Path("../../Atlas_2/Training/Cropped_128_Combined"),
        Path("./Atlas_2/Training/Cropped_128_Combined"),
    ]
    data_dir_found = next((p for p in fallback_paths if _has_flat_combined_dir(p)), None)
    if data_dir_found is None:
        print("❌ No valid data directory found for quick discovery.")
        pairs = []
        lesion_presence = []
    else:
        print(f"✅ Quick discovery using: {data_dir_found}")
        images = sorted(data_dir_found.glob("*nii_T1w_cropped128.nii.gz"))
        masks = sorted(data_dir_found.glob("*nii_mask_cropped128.nii.gz"))
        def _basekey(name: str) -> str:
            if name.endswith("nii_T1w_cropped128.nii.gz"):
                return name[:-len("nii_T1w_cropped128.nii.gz")]
            if name.endswith("nii_mask_cropped128.nii.gz"):
                return name[:-len("nii_mask_cropped128.nii.gz")]
            return name.split(".nii")[0]
        image_map = {_basekey(p.name): p for p in images}
        mask_map = {_basekey(p.name): p for p in masks}
        keys = sorted(set(image_map) & set(mask_map))
        pairs = [(image_map[k], mask_map[k]) for k in keys]
        lesion_presence = [1]*len(pairs)
        print(f"✅ Paired {len(pairs)} samples via quick discovery")

print("🚀 Starting comprehensive testing on REAL Atlas data...")
print(f"📊 Model ready: {type(m)}")
print(f"📚 Dataset ready: {len(pairs)} pairs" if 'pairs' in globals() else "📚 Dataset not available")
print("🎯 Testing ONLY on real Atlas data (no synthetic data)")

# Run comprehensive test on real Atlas data if we have pairs
results = run_comprehensive_test(model=m, pairs=pairs) if 'pairs' in globals() and len(pairs) > 0 else None

if results is None:
    print("⚠️ Skipped testing because no dataset pairs were found. Run the dataset preparation cell above.")
else:
    print("\n🎉 COMPREHENSIVE TESTING COMPLETED!")
    print("📁 All results saved to test_results/ directory")
    print("🔍 Check test_results/ for:")
    print("   • JSON results file")
    print("   • Individual prediction .npy files") 
    print("   • Visualization .png files")

⚠️ 'pairs' not found. Attempting quick dataset discovery...
✅ Quick discovery using: /home/rbielski/Atlas_2/Training/Cropped_128_Combined
✅ Paired 655 samples via quick discovery
🚀 Starting comprehensive testing on REAL Atlas data...
📊 Model ready: <class 'keras.src.engine.functional.Functional'>
📚 Dataset ready: 655 pairs
🎯 Testing ONLY on real Atlas data (no synthetic data)
🚀 Starting comprehensive test on 655 samples...
📊 Processing sample 1/655: sub-r001s001_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz
  ✅ Dice: 0.8358, IoU: 0.7179, Vol Error: 0.2228
📊 Processing sample 2/655: sub-r001s002_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz
  ✅ Dice: 0.8358, IoU: 0.7179, Vol Error: 0.2228
📊 Processing sample 2/655: sub-r001s002_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz
  ✅ Dice: 0.8886, IoU: 0.7996, Vol Error: 0.0974
📊 Processing sample 3/655: sub-r001s003_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.nii.gz
  ✅ Dice: 0.8886, IoU: 0.7996, Vol Err

In [7]:
# ===================================================================
# 🎨 INTERACTIVE VISUALIZATION VIEWER
# ===================================================================

print("🎨 Setting up interactive visualization viewer...")

# Self-contained imports for robustness
from pathlib import Path
import ipywidgets as widgets
import nibabel as nib
import numpy as np

# Resolve target shape from model/test pipeline
TARGET_SHAPE = tuple(MODEL_INPUT_SHAPE) if 'MODEL_INPUT_SHAPE' in globals() else (128, 128, 128)

class InteractiveStrokeViewer:
    """Interactive viewer for stroke segmentation results."""
    
    def __init__(self, model, pairs):
        self.model = model
        self.pairs = pairs or []
        self.current_data = None
        self.current_sample_idx = 0

        if len(self.pairs) == 0:
            print("⚠️ No pairs available for visualization.")
            return
        
        # Create widgets (use model-derived depth for slider)
        mid_slice = max(0, min(TARGET_SHAPE[2] // 2, TARGET_SHAPE[2] - 1))
        self.sample_dropdown = widgets.Dropdown(
            options=[(f"Sample {i}: {Path(self.pairs[i][0]).stem}", i) for i in range(min(len(self.pairs), 10))],
            value=0,
            description='Sample:',
            style={'description_width': 'initial'}
        )
        
        self.slice_slider = widgets.IntSlider(
            value=mid_slice,
            min=0,
            max=max(0, TARGET_SHAPE[2]-1),
            step=1,
            description='Slice:',
            style={'description_width': 'initial'}
        )
        
        self.threshold_slider = widgets.FloatSlider(
            value=0.5,
            min=0.0,
            max=1.0,
            step=0.05,
            description='Threshold:',
            style={'description_width': 'initial'}
        )
        
        self.alpha_slider = widgets.FloatSlider(
            value=0.6,
            min=0.0,
            max=1.0,
            step=0.1,
            description='Overlay Alpha:',
            style={'description_width': 'initial'}
        )
        
        self.metrics_output = widgets.Output()
        self.plot_output = widgets.Output()
        
        # Layout
        controls = widgets.VBox([
            self.sample_dropdown,
            self.slice_slider,
            self.threshold_slider,
            self.alpha_slider
        ])
        
        self.ui = widgets.VBox([
            widgets.HTML("<h3>🧠 Interactive Stroke Segmentation Viewer</h3>"),
            controls,
            self.metrics_output,
            self.plot_output
        ])
        
        # Set up event handlers
        self.sample_dropdown.observe(self.on_sample_change, names='value')
        self.slice_slider.observe(self.on_param_change, names='value')
        self.threshold_slider.observe(self.on_param_change, names='value')
        self.alpha_slider.observe(self.on_param_change, names='value')
        
        # Load initial sample
        self.load_sample(0)
    
    def load_sample(self, sample_idx):
        """Load and process a sample."""
        try:
            img_path, mask_path = self.pairs[sample_idx]
            
            # Load and preprocess image
            img_volume = preprocess_volume(img_path, TARGET_SHAPE)
            
            # Load ground truth mask
            mask_nii = nib.load(str(mask_path))
            mask_volume = mask_nii.get_fdata()
            if mask_volume.ndim == 4:
                mask_volume = mask_volume[:, :, :, 0]
            
            # Resize mask to model input
            mask_resized = robust_resize_with_context(mask_volume, TARGET_SHAPE)
            
            # Run prediction
            prediction = self.model.predict(img_volume, verbose=0)
            
            self.current_data = {
                'image': img_volume[0, ..., 0],
                'ground_truth': (mask_resized > 0.5).astype(np.float32),
                'prediction_raw': prediction[0, ..., 0],
                'sample_idx': sample_idx,
                'img_name': Path(img_path).name,
                'mask_name': Path(mask_path).name
            }
            
            # Update slice slider max based on actual image depth
            self.slice_slider.max = max(0, self.current_data['image'].shape[2] - 1)
            self.slice_slider.value = min(self.slice_slider.value, self.slice_slider.max)
            
            self.update_display()
            
        except Exception as e:
            with self.metrics_output:
                from IPython.display import clear_output
                clear_output(wait=True)
                print(f"❌ Error loading sample {sample_idx}: {e}")
    
    def on_sample_change(self, change):
        """Handle sample selection change."""
        self.load_sample(change['new'])
    
    def on_param_change(self, change):
        """Handle parameter changes."""
        if self.current_data is not None:
            self.update_display()
    
    def update_display(self):
        """Update the visualization."""
        if self.current_data is None:
            return
        
        data = self.current_data
        slice_idx = self.slice_slider.value
        threshold = self.threshold_slider.value
        alpha = self.alpha_slider.value
        
        # Get current slice
        img_slice = data['image'][:, :, slice_idx]
        gt_slice = data['ground_truth'][:, :, slice_idx]
        pred_raw_slice = data['prediction_raw'][:, :, slice_idx]
        pred_slice = (pred_raw_slice > threshold).astype(np.float32)
        
        # Calculate metrics for current slice
        dice_coefficient_np_local = dice_coefficient_np  # from pipeline cell
        iou_score_np_local = iou_score_np               # from pipeline cell
        slice_dice = dice_coefficient_np_local(gt_slice, pred_slice)
        slice_iou = iou_score_np_local(gt_slice, pred_slice)
        
        # Calculate volume metrics
        volume_dice = dice_coefficient_np_local(data['ground_truth'], (data['prediction_raw'] > threshold).astype(np.float32))
        volume_iou = iou_score_np_local(data['ground_truth'], (data['prediction_raw'] > threshold).astype(np.float32))
        
        # Update metrics display
        with self.metrics_output:
            from IPython.display import clear_output
            clear_output(wait=True)
            print(f"📊 Sample: {data['img_name']}")
            print(f"🔍 Slice {slice_idx}/{data['image'].shape[2]-1}")
            print(f"📈 Slice Metrics - Dice: {slice_dice:.4f}, IoU: {slice_iou:.4f}")
            print(f"📈 Volume Metrics - Dice: {volume_dice:.4f}, IoU: {volume_iou:.4f}")
            print(f"🎚️ Threshold: {threshold:.2f}, Alpha: {alpha:.1f}")
        
        # Update plot
        with self.plot_output:
            from IPython.display import clear_output
            import matplotlib.pyplot as plt
            clear_output(wait=True)
            fig, axes = plt.subplots(2, 2, figsize=(12, 10))
            axes[0, 0].imshow(img_slice, cmap='gray')
            axes[0, 0].set_title('Original Image')
            axes[0, 0].axis('off')
            axes[0, 1].imshow(img_slice, cmap='gray')
            if np.any(gt_slice):
                axes[0, 1].imshow(gt_slice, cmap='Reds', alpha=alpha)
            axes[0, 1].set_title(f'Ground Truth (volume: {np.sum(data["ground_truth"]):.0f})')
            axes[0, 1].axis('off')
            axes[1, 0].imshow(img_slice, cmap='gray')
            if np.any(pred_slice):
                axes[1, 0].imshow(pred_slice, cmap='Blues', alpha=alpha)
            axes[1, 0].set_title(f'Prediction (volume: {np.sum(pred_slice):.0f})')
            axes[1, 0].axis('off')
            im = axes[1, 1].imshow(pred_raw_slice, cmap='viridis', vmin=0, vmax=1)
            axes[1, 1].set_title('Prediction Heatmap')
            axes[1, 1].axis('off')
            plt.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)
            plt.tight_layout()
            plt.show()
    
    def display(self):
        if len(self.pairs) == 0:
            return
        from IPython.display import display
        display(self.ui)

# Create and display the interactive viewer (guarded)
if 'pairs' in globals() and isinstance(pairs, (list, tuple)) and len(pairs) > 0:
    print("🎯 Creating interactive viewer...")
    viewer = InteractiveStrokeViewer(m, pairs)
    if isinstance(viewer, InteractiveStrokeViewer) and len(pairs) > 0:
        print("✅ Interactive viewer ready!")
        print("📋 Features:")
        print("   • Dropdown to select different samples")
        print("   • Slice slider to navigate through 3D volume")
        print("   • Threshold slider to adjust prediction sensitivity") 
        print("   • Alpha slider to adjust overlay transparency")
        print("   • Real-time metrics calculation")
        print("   • 4-panel visualization (original, GT, prediction, heatmap)")
        viewer.display()
else:
    print("⚠️ Skipping viewer: no dataset pairs available.")

🎨 Setting up interactive visualization viewer...
🎯 Creating interactive viewer...


✅ Interactive viewer ready!
📋 Features:
   • Dropdown to select different samples
   • Slice slider to navigate through 3D volume
   • Threshold slider to adjust prediction sensitivity
   • Alpha slider to adjust overlay transparency
   • Real-time metrics calculation
   • 4-panel visualization (original, GT, prediction, heatmap)


In [8]:
# ===================================================================
# 📋 FINAL SUMMARY - DETAILED, PATH-CORRECT, ACTIONABLE
# ===================================================================
import os, json, glob, math
from pathlib import Path
import numpy as np

print("🎯 Generating final summary...")

# Resolve paths
tr_dir = Path(test_results_dir).resolve() if 'test_results_dir' in globals() else Path('test_results').resolve()
ds_dir = str(getattr(globals().get('config', object()), 'DATA_DIR', 'unknown'))

# Best-guess model path (pick most recent existing candidate if list available)
best_guess_model_path = None
try:
    if 'model_candidates' in globals():
        existing = [p for p in model_candidates if hasattr(p, 'exists') and p.exists()]
        if existing:
            best_guess_model_path = str(sorted(existing, key=lambda p: p.stat().st_mtime, reverse=True)[0])
except Exception:
    pass

# Load results (prefer in-memory 'results', else pick newest JSON on disk)
loaded_results = None
if 'results' in globals() and isinstance(results, dict) and results.get('sample_results'):
    loaded_results = results
else:
    tr_dir.mkdir(exist_ok=True)
    json_files = sorted(tr_dir.glob('test_results_*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if json_files:
        try:
            with open(json_files[0], 'r') as f:
                loaded_results = json.load(f)
        except Exception as e:
            print(f"⚠️ Could not load results file {json_files[0]}: {e}")

# Derive metrics
total_pairs = len(pairs) if 'pairs' in globals() else 0
lesion_presence_ratio = None
if 'lesion_presence' in globals() and hasattr(lesion_presence, '__len__') and len(lesion_presence) > 0:
    try:
        lesion_presence_ratio = float(np.mean(lesion_presence))
    except Exception:
        lesion_presence_ratio = None

model_info = {
    'input_shape': str(getattr(globals().get('m', object()), 'input_shape', 'unknown')),
    'output_shape': str(getattr(globals().get('m', object()), 'output_shape', 'unknown')),
    'parameters': int(globals().get('m').count_params()) if 'm' in globals() else 'unknown'
}

summary = None
sample_results = []
if loaded_results:
    sample_results = loaded_results.get('sample_results', [])
    summary = loaded_results.get('summary')

# Compute additional diagnostics if not present
def _safe(v, d=0.0):
    try: return float(v)
    except: return d

valid = [r for r in sample_results if 'error' not in r]
dice_scores = [_safe(r.get('dice_score')) for r in valid] if valid else []
iou_scores = [_safe(r.get('iou_score')) for r in valid] if valid else []
gt_vols = [_safe(r.get('gt_volume')) for r in valid] if valid else []
pred_vols = [_safe(r.get('pred_volume')) for r in valid] if valid else []

mean_dice = float(np.mean(dice_scores)) if dice_scores else None
std_dice = float(np.std(dice_scores)) if dice_scores else None
mean_iou = float(np.mean(iou_scores)) if iou_scores else None
std_iou = float(np.std(iou_scores)) if iou_scores else None
mean_gt_vol = float(np.mean(gt_vols)) if gt_vols else None
mean_pred_vol = float(np.mean(pred_vols)) if pred_vols else None
vol_bias = (mean_pred_vol - mean_gt_vol) if (mean_gt_vol is not None and mean_pred_vol is not None) else None

# Top failures and wins
top_worst = []
top_best = []
if valid and dice_scores:
    sorted_valid = sorted(valid, key=lambda r: r.get('dice_score', 0.0))
    top_worst = sorted_valid[:min(5, len(sorted_valid))]
    top_best = sorted_valid[-min(5, len(sorted_valid)):]

# Print core summary
print("="*70)
print("📁 Paths")
print(f"• Test results dir: {tr_dir}")
print(f"• Dataset dir:      {ds_dir}")
print(f"• Model file:       {best_guess_model_path or 'unknown'}")

print("\n🧠 Model")
print(f"• Parameters: {model_info['parameters']}")
print(f"• Input:      {model_info['input_shape']}")
print(f"• Output:     {model_info['output_shape']}")

print("\n📚 Dataset")
print(f"• Pairs available: {total_pairs}")
if lesion_presence_ratio is not None:
    print(f"• Lesion presence: {lesion_presence_ratio*100:.1f}%")

if loaded_results and valid:
    print("\n📈 Metrics (valid samples)")
    print(f"• Mean Dice: {mean_dice:.4f} ± {std_dice:.4f}")
    print(f"• Mean IoU:  {mean_iou:.4f} ± {std_iou:.4f}")
    print(f"• Valid/Total: {len(valid)}/{loaded_results.get('total_samples', len(sample_results))}")
    if mean_gt_vol is not None:
        print(f"• Mean GT voxels:   {int(mean_gt_vol)}")
    if mean_pred_vol is not None:
        print(f"• Mean Pred voxels: {int(mean_pred_vol)}")
    if vol_bias is not None:
        sign = 'over' if vol_bias > 0 else 'under'
        print(f"• Volume bias: {sign}-segmentation by {abs(vol_bias):.1f} voxels on average")
else:
    print("\n⚠️ No results found. Run testing cell before summary.")

# Show worst and best cases
def _name(r):
    return r.get('image_file') or r.get('sample_id')

if top_worst:
    print("\n🔻 Lowest Dice cases (up to 5)")
    for r in top_worst:
        print(f"  • {str(_name(r))[:64]:<64}  Dice={r.get('dice_score',0):.4f}  IoU={r.get('iou_score',0):.4f}  GT={int(_safe(r.get('gt_volume')))} Pred={int(_safe(r.get('pred_volume')))}")
if top_best:
    print("\n🔺 Highest Dice cases (up to 5)")
    for r in reversed(top_best):
        print(f"  • {str(_name(r))[:64]:<64}  Dice={r.get('dice_score',0):.4f}  IoU={r.get('iou_score',0):.4f}")

# Actionable recommendations
print("\n🛠️ Recommendations for next training run")
recs = []

# 1) Boundary vs region balance
if mean_dice is not None and mean_dice < 0.75:
    recs.append("Increase boundary_weighted_loss weight (e.g., DICE:BOUNDARY from 0.5:0.5 → 0.4:0.6)")
else:
    recs.append("Keep current loss balance; focus on data diversity and regularization fine-tuning")

# 2) Small lesion emphasis
if mean_gt_vol is not None and mean_gt_vol < 3000:
    recs.append("Boost small lesion handling: lower SMALL_LESION_THRESHOLD and increase enhancement iterations")
    recs.append("Slightly raise SYNTHETIC_LESION_PROB to expose more tiny targets")

# 3) Volume bias correction
if vol_bias is not None and abs(vol_bias) > 0:
    if vol_bias > 0:
        recs.append("Model over-segments: raise decision threshold during training eval or add precision-focused augmentation")
    else:
        recs.append("Model under-segments: add recall-focused augmentation and increase positive class emphasis")

# 4) Augmentation and regularization
recs.append("Slightly increase augmentation intensity (rotations, flips, gamma) if overfitting observed")
recs.append("Tune L2_REG and DROPOUT_RATE inversely with batch size adjustments")

# 5) Curriculum and sampling
if lesion_presence_ratio is not None and lesion_presence_ratio < 0.7:
    recs.append("If negatives dominate, adopt class-aware sampling or hard-example mining")

# 6) LR schedule
recs.append("Consider longer warmup or lower MIN_LR if late-epoch plateaus are observed")

for i, r in enumerate(recs, 1):
    print(f"{i}. {r}")

print("\n✅ Summary complete.")

🎯 Generating final summary...
📁 Paths
• Test results dir: /home/rbielski/stroke_cleaned/stroke_segmentation_v1.1_success/test_results
• Dataset dir:      unknown
• Model file:       /home/rbielski/stroke_cleaned/stroke_segmentation_v1.1_success/models/emergency_cropped_save_20250721_193525.keras

🧠 Model
• Parameters: 11537305
• Input:      (None, 128, 128, 128, 1)
• Output:     (None, 128, 128, 128, 1)

📚 Dataset
• Pairs available: 655
• Lesion presence: 100.0%

📈 Metrics (valid samples)
• Mean Dice: 0.4735 ± 0.3471
• Mean IoU:  0.3777 ± 0.2996
• Valid/Total: 655/655
• Mean GT voxels:   31081
• Mean Pred voxels: 33097
• Volume bias: over-segmentation by 2015.8 voxels on average

🔻 Lowest Dice cases (up to 5)
  • sub-r023s001_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.n  Dice=0.0000  IoU=0.0000  GT=2563 Pred=37719
  • sub-r038s081_ses-1_space-MNI152NLin2009aSym.nii_T1w_cropped128.n  Dice=0.0000  IoU=0.0000  GT=30602 Pred=0
  • sub-r001s022_ses-1_space-MNI152NLin2009aSym.nii_T1w_